# 📐 CareFlow — Notebook 4: Statistical Hypothesis Testing
## Pearson Correlations · t-tests · ANOVA · Chi-Square · SPC Control Charts

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import (
    shapiro, normaltest, levene, ttest_ind, mannwhitneyu,
    kruskal, f_oneway, chi2_contingency, pearsonr, spearmanr
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
COLORS = ['#06b6d4','#8b5cf6','#10b981','#f59e0b','#f43f5e','#3b82f6']
ALPHA = 0.05

# Dataset generation
np.random.seed(42)
N = 1000
acuity = np.random.choice([1,2,3,4,5], N, p=[0.018,0.045,0.462,0.341,0.134])
ages = np.where(np.random.rand(N)<0.35, np.random.normal(34,12,N), np.random.normal(68,14,N)).clip(0,100).round(0).astype(int)
los_base = {1:480, 2:320, 3:218, 4:112, 5:64}
los = np.array([max(10, np.random.normal(los_base[a], los_base[a]*0.35)) for a in acuity]).round(1)
loopback = np.random.choice([0,1], N, p=[0.602,0.398])
cpoe = np.random.choice([0,1], N, p=[0.688,0.312])
gender = np.random.choice(['Male','Female'], N)
is_weekend = np.random.choice([0,1], N, p=[0.714,0.286])
readmit = np.where(acuity <= 2, np.random.choice([0,1],N,p=[0.65,0.35]), np.random.choice([0,1],N,p=[0.88,0.12]))

df = pd.DataFrame({'acuity':acuity,'age':ages,'los':los,'loopback':loopback,
                    'cpoe':cpoe,'gender':gender,'is_weekend':is_weekend,'readmit_30d':readmit})
print(f'✅ Dataset ready: {df.shape}')

In [ ]:
# ─── HYPOTHESIS TESTING BATTERY ──────────────────────────────────────────────

def print_test(title, stat, p, df_val=None, reject_label='H0 REJECTED', accept_label='H0 NOT REJECTED'):
    decision = reject_label if p < ALPHA else accept_label
    print(f'\n  ┌─ {title}')
    print(f'  │  Statistic = {stat:.4f}, p = {p:.4e}')
    if df_val: print(f'  │  df = {df_val}')
    print(f'  └─ {decision} at α={ALPHA} → {"✅ Significant" if p < ALPHA else "⬜ Not Significant"}')
    return p < ALPHA

print('═'*65)
print('STATISTICAL HYPOTHESIS TESTING — CareFlow ED Analysis')
print('═'*65)

# ── Test 1: Normality ──────────────────────────────────────────────────────────
print('\n[1] NORMALITY TESTS (LOS Distribution)')
_, p_sw = shapiro(df['los'].sample(500, random_state=42))
print_test('Shapiro-Wilk (n=500 sample)', _, p_sw)

stat_da, p_da = normaltest(df['los'])
print_test("D'Agostino-Pearson (n=1000)", stat_da, p_da)

print(f'  → LOS Skewness: {df["los"].skew():.3f} | Kurtosis: {df["los"].kurtosis():.3f}')
print(f'  → Conclusion: LOS is NON-NORMAL (right-skewed). Use non-parametric tests.')

In [ ]:
# ── Test 2: Loopback vs Direct LOS (Welch t-test + Mann-Whitney U) ────────────
print('[2] LOOPBACK vs DIRECT LOS')
los_loop = df[df['loopback']==1]['los']
los_direct = df[df['loopback']==0]['los']

# Levene's test for equal variances
lev_stat, lev_p = levene(los_loop, los_direct)
print_test('Levene Variance Homogeneity', lev_stat, lev_p,
            reject_label='Unequal variances → Use Welch t-test')

t, pt = ttest_ind(los_loop, los_direct, equal_var=False)
print_test('Welch t-test (two-tailed)', t, pt)

u, pu = mannwhitneyu(los_loop, los_direct, alternative='two-sided')
print_test('Mann-Whitney U test (non-parametric)', u, pu)

print(f'  Loopback: mean={los_loop.mean():.1f} min (n={len(los_loop)})')
print(f'  Direct:   mean={los_direct.mean():.1f} min (n={len(los_direct)})')
print(f'  Effect size (Cohen d) = {(los_loop.mean()-los_direct.mean())/np.sqrt((los_loop.std()**2+los_direct.std()**2)/2):.3f}')

In [ ]:
# ── Test 3: LOS across ESI Acuity Levels (Kruskal-Wallis + one-way ANOVA) ─────
print('[3] LOS ACROSS ESI ACUITY LEVELS')
groups = [df[df['acuity']==i]['los'].values for i in [1,2,3,4,5]]

f_stat, pf = f_oneway(*groups)
print_test('One-way ANOVA', f_stat, pf, df_val=f'{4}, {N-5}')

h_stat, ph = kruskal(*groups)
print_test('Kruskal-Wallis H test (non-parametric)', h_stat, ph)

print('  Group means:')
for i, g in zip([1,2,3,4,5], groups):
    print(f'    ESI-{i}: mean={np.mean(g):.1f} min (n={len(g)})')

In [ ]:
# ── Test 4: CPOE vs Paper Requisition LOS Reduction ───────────────────────────
print('[4] CPOE vs PAPER: LOS Difference')
los_cpoe = df[df['cpoe']==1]['los']
los_paper = df[df['cpoe']==0]['los']
t4, p4 = ttest_ind(los_cpoe, los_paper, equal_var=False)
print_test('Welch t-test: CPOE vs Paper', t4, p4)
print(f'  CPOE mean LOS:  {los_cpoe.mean():.1f} min')
print(f'  Paper mean LOS: {los_paper.mean():.1f} min')
print(f'  LOS reduction:  {los_paper.mean() - los_cpoe.mean():.1f} min ({(los_paper.mean()-los_cpoe.mean())/los_paper.mean()*100:.1f}%)')

# ── Test 5: Chi-square: Gender × Readmission ───────────────────────────────────
print('[5] CHI-SQUARE: Gender × Readmission')
contingency = pd.crosstab(df['gender'], df['readmit_30d'])
print('  Contingency table:')
print(contingency)
chi2, p_chi2, ddof, expected = chi2_contingency(contingency)
print_test(f'Chi-square test (df={ddof})', chi2, p_chi2, df_val=ddof)

# Cramér's V
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
print(f'  Cramér\'s V (effect size) = {cramers_v:.4f} → {"weak" if cramers_v < 0.1 else "moderate" if cramers_v < 0.3 else "strong"} association')

In [ ]:
# ── Test 6: Correlation Tests ──────────────────────────────────────────────────
print('[6] CORRELATION ANALYSIS')
pairs = [
    ('age', 'los', 'Age vs LOS'),
    ('acuity', 'los', 'Acuity vs LOS'),
    ('loopback', 'los', 'Loopback vs LOS'),
]
for col1, col2, label in pairs:
    r_p, p_p = pearsonr(df[col1], df[col2])
    r_s, p_s = spearmanr(df[col1], df[col2])
    print(f'\n  {label}:')
    print(f'    Pearson r  = {r_p:.4f} (p={p_p:.4e})')
    print(f'    Spearman ρ = {r_s:.4f} (p={p_s:.4e})')
    print(f'    Significant: {"Yes" if p_p < ALPHA else "No"}')

In [ ]:
# ─── Statistical Power & Sample Size Calculation ──────────────────────────────
from scipy.stats import norm

def sample_size_ttest(effect_size, alpha=0.05, power=0.80):
    """Calculate minimum sample size for two-sample t-test."""
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    n = 2 * ((z_alpha + z_beta) / effect_size) ** 2
    return int(np.ceil(n))

print('═'*55)
print('STATISTICAL POWER ANALYSIS')
print('═'*55)
for effect, label in [(0.2,'Small (d=0.2)'), (0.5,'Medium (d=0.5)'), (0.8,'Large (d=0.8)')]:
    n = sample_size_ttest(effect)
    print(f'  {label}: n={n} per group (α=0.05, power=80%)')

print(f'\n  Our study n=1000 → Highly powered for medium to large effects')
print('\n✅ Statistical Testing Notebook Complete!')

In [ ]:
# ─── SPC Control Chart for LOS ────────────────────────────────────────────────
# Simulate daily LOS averages over 30 days
np.random.seed(99)
daily_los = []
for day in range(30):
    day_sample = np.random.normal(218, 32, np.random.randint(28,52))
    # Add special cause variation on day 14 and 22
    if day in [14, 22]: day_sample += np.random.normal(45, 5)
    daily_los.append({'day': day+1, 'mean_los': day_sample.mean(), 'n': len(day_sample)})

spc_df = pd.DataFrame(daily_los)
CL = spc_df['mean_los'].mean()
sigma_est = spc_df['mean_los'].std()
UCL = CL + 3 * sigma_est
LCL = max(0, CL - 3 * sigma_est)

violations = spc_df[spc_df['mean_los'] > UCL]

plt.figure(figsize=(16, 5))
plt.plot(spc_df['day'], spc_df['mean_los'], 'o-', color='#06b6d4', linewidth=2, markersize=5, label='Daily Avg LOS')
plt.axhline(CL, color='white', linestyle='--', linewidth=1.5, label=f'CL={CL:.1f} min')
plt.axhline(UCL, color='#f43f5e', linestyle='--', linewidth=1.5, label=f'UCL={UCL:.1f} min')
plt.axhline(LCL, color='#10b981', linestyle='--', linewidth=1.5, label=f'LCL={LCL:.1f} min')
plt.fill_between(spc_df['day'], LCL, UCL, alpha=0.07, color='white')

# Mark violations
plt.scatter(violations['day'], violations['mean_los'], color='#f43f5e', s=80, zorder=5,
            label=f'UCL violations ({len(violations)})', marker='x', linewidths=2)

plt.title('Shewhart X-bar Control Chart — Daily Average LOS (30 Days)', fontsize=13, color='white')
plt.xlabel('Day of Month', color='white')
plt.ylabel('Avg LOS (minutes)', color='white')
plt.legend(loc='upper right', fontsize=9)
plt.tick_params(colors='white')
plt.tight_layout()
plt.savefig('../docs/stats_fig1_spc_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ SPC chart saved — {len(violations)} UCL violations detected on days: {violations["day"].tolist()}')